# Laya phishing fine-tuning v2

A controlled 1,024-token baseline using cleaned HTML, cross-entropy, lower learning rates,
and 10% warmup. Targets 10,000 training, 2,000 validation and 2,000 test pages, balanced by label,
using up to 16 evenly spread training shards and 8 test shards. Deduplication and host isolation may
reduce actual counts; inspect the sampling report. These are new splits, not the original baseline splits. This is phishing versus benign, not a detector of all malware.

Run cells in order on a CUDA GPU (Python 3.11/3.12 recommended). No training outputs are precomputed.
Artifacts go to `laya-phishing-v2`; the original notebook and existing artifacts are untouched.

Validation selects the checkpoint, temperature, and decision threshold; test is scored only after
those choices are fixed. Reusing validation for these choices is a small-data compromise, not an
independent calibration evaluation. With a target of 1,000 benign validation pages, empirical FPR resolution is 0.1 percentage
point if the target is reached. The default 2% empirical FPR target is exploratory, not a production guarantee.

For a subsequent data experiment: expand real benign login/payment pages and short pages, group by
registered domain and template, and compare compact HTML at 1,024/2,048 tokens. Those changes are
intentionally separate from this expanded-data run. The sampler here groups exact hostnames.

In [ ]:
%pip install -q "laya>=0.3.0" "transformers>=4.48.0" safetensors huggingface_hub pyarrow scikit-learn beautifulsoup4

In [ ]:
import torch, transformers, laya
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator")

## 1. Configuration

Sampling constants are `train_html_phishing_classifier.ipynb`'s, unchanged, so the split
is the one MarkupLM `v1` trained on. Learning rates come from Laya's own fine-tuning
recipe: the encoder moves slowly, the freshly-shaped head faster.

`MICRO_BATCH` is deliberately small. ModernBERT-large is 421M parameters against
MarkupLM-base's 136M, and a T4 has 16 GB. Raise it if `nvidia-smi` shows headroom; lower
it to 2 if you hit an out-of-memory error.


In [ ]:
import json, math, random, hashlib, re, os, time
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlsplit

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from importlib.metadata import version

SEED = 42
# The sampling cell below is copied verbatim from the MarkupLM notebook and
# expects this mapping, which lives in that notebook's imports cell.
PHRESHPHISH_LABEL_TO_ID = {"benign": 0, "phish": 1}
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

BASE_REPO = "convaiinnovations/laya"          # English checkpoint, ModernBERT-large
ARTIFACT_DIR = Path("laya-phishing-v2")        # written at the end, download from Kaggle output

# Conservative cross-entropy baseline; tune on validation, never test.
ENCODER_LR = 1.0e-5
HEAD_LR = 5.0e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
MICRO_BATCH = 2
GRAD_ACCUM = 8                                 # effective batch 16 on one GPU
MAX_LEN = 1024                                 # total sequence, including question/options
HEAD_MAX_LEN = 192
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
TARGET_FPR = 0.02
TEMPERATURE_MIN, TEMPERATURE_MAX = 0.5, 5.0  # match the Agent runtime clamp

# The question the model is being taught to answer. Index 1 is "yes" (laya/agent.py).
QUESTION = {"t": "noul",
            "ins": ("Is this web page a phishing page that impersonates a real brand "
                    "to steal user credentials?"),
            "crit": None}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This notebook needs a GPU. On Kaggle: Settings > Accelerator > GPU.")
# T4 is Turing and has no bfloat16; A100/P100 differ, so ask rather than assume.
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", torch.cuda.get_device_name(0), "| amp:", AMP_DTYPE)
# Set this to a saved last.pt to resume at the next completed-epoch boundary.
# Leave None for a new run. Do not change hyperparameters or datasets when resuming.
RESUME_FROM = None  # e.g. Path("/kaggle/input/my-checkpoints/last.pt")
CHECKPOINT_ROOT = Path("laya-phishing-v2-checkpoints")


## 2. Expanded dataset coverage

Targets: 5,000 training, 1,000 validation and 1,000 test pages **per class**.
Scan up to 16 training shards and 8 test shards, evenly spaced through the sorted shard list.
Set either shard count to `None` to scan every shard of that split; training still uses a bounded sample.
Larger shard coverage downloads and scans full parquet files, so allow substantial disk space and time.

A separate `laya_v2_expanded` cache prevents accidental reuse or overwrite of the original small sample.
Cache configuration includes sample targets and shard counts; changing them rebuilds the candidate pools.
Training short-page and benign-login enrichment is preserved. Sample caps are targets, not guaranteed
counts after deduplication and exact-host isolation. Check printed warnings and the saved sampling report.
Registered-domain/template isolation remains a future improvement.

At the same effective batch size this target gives roughly 12.5 times as many optimizer updates per epoch
as the previous 800-page run. Start with the existing early stopping; do not increase epochs automatically.


In [ ]:
DATASET_ID = "phreshphish/phreshphish"
CACHE_DIR = Path("datasets/phreshphish_sample/laya_v2_expanded")
USE_TRAIN_PER_CLASS = 5_000
USE_VAL_PER_CLASS = 1_000
USE_TEST_PER_CLASS = 1_000
TRAIN_SHARDS = 16  # evenly spread across available shards; None scans all
TEST_SHARDS = 8   # evenly spread across available shards; None scans all
MIN_HTML_CHARS = 20
MAX_HTML_CHARS = 1_000_000
PARSE_CHAR_BUDGET = 300_000
SHORT_HTML_CHARS = 2_000
SHORT_FRACTION = 0.20
BENIGN_LOGIN_FRACTION = 0.20
LOGIN_RE = re.compile(r"<input\b[^>]*\btype\s*=\s*['\"]?password\b", re.I)


def normalize_page(row):
    html = row.get("html") or ""
    label = row.get("label")
    if label not in PHRESHPHISH_LABEL_TO_ID or not MIN_HTML_CHARS <= len(html) <= MAX_HTML_CHARS:
        return None
    url = row.get("url") or ""
    try:
        host = (urlsplit(url).hostname or "").lower().rstrip(".")
    except ValueError:
        host = ""
    # Hash before truncation, so different tails are not mistaken for identical pages.
    digest = hashlib.sha256(html.encode("utf-8", "ignore")).hexdigest()
    return {"html": html[:PARSE_CHAR_BUDGET], "html_len": len(html),
            "label": label, "label_id": PHRESHPHISH_LABEL_TO_ID[label], "url": url,
            "host": host, "sha256": digest,
            "is_login": bool(LOGIN_RE.search(html)), "short": len(html) <= SHORT_HTML_CHARS}


def host_group(row):
    return "host:" + row["host"] if row["host"] else "hash:" + row["sha256"]


def reservoir_add(bucket, row, count, capacity, rng):
    if len(bucket) < capacity:
        bucket.append(row)
    else:
        position = rng.randrange(count)
        if position < capacity:
            bucket[position] = row


def select_coverage(uniform, short, login, limit, short_fraction, login_fraction):
    selected, seen = [], set()
    def add(rows, target):
        added = 0
        for row in rows:
            if len(selected) >= limit or added >= target:
                break
            if row["sha256"] not in seen:
                selected.append(row)
                seen.add(row["sha256"])
                added += 1
    add(short, round(limit * short_fraction))
    add(login, round(limit * login_fraction))
    add(uniform, limit)
    return selected


def sample_shard(local_path, per_class, seed, enrich):
    rng = random.Random(seed)
    pools = {(label, kind): [] for label in (0, 1) for kind in ("uniform", "short", "login")}
    counts = Counter()
    for batch in pq.ParquetFile(local_path).iter_batches(batch_size=64, columns=["html", "url", "label"]):
        for raw in batch.to_pylist():
            row = normalize_page(raw)
            if row is None:
                continue
            label = row["label_id"]
            kinds = ["uniform"]
            if enrich and row["short"]:
                kinds.append("short")
            if enrich and label == 0 and row["is_login"]:
                kinds.append("login")
            for kind in kinds:
                key = (label, kind)
                counts[key] += 1
                reservoir_add(pools[key], row, counts[key], per_class, rng)
    selected = []
    for label in (0, 1):
        for kind in ("uniform", "short", "login"):
            rng.shuffle(pools[(label, kind)])
        selected += select_coverage(
            pools[(label, "uniform")], pools[(label, "short")], pools[(label, "login")],
            per_class, SHORT_FRACTION if enrich else 0,
            BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0,
        )
    return selected


def select_shard_indices(total, requested):
    if total < 1:
        raise ValueError("No shards available")
    if requested is not None and (not isinstance(requested, int) or isinstance(requested, bool) or requested < 1):
        raise ValueError("Shard count must be a positive integer or None for all")
    count = total if requested is None else min(requested, total)
    return [round(i * (total - 1) / max(count - 1, 1)) for i in range(count)]


def sample_split(split, per_class, shard_count, enrich):
    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    if not files:
        raise RuntimeError(f"No parquet shards for {split}")
    indices = select_shard_indices(len(files), shard_count)
    count = len(indices)
    print(f"{split}: scanning {count}/{len(files)} shards; candidate target {per_class:,} per class")
    print("Whole selected shards are downloaded and scanned; only sampled rows enter training.")
    rows = []
    for index in indices:
        shard = files[index]
        local = hf_hub_download(DATASET_ID, shard, repo_type="dataset")
        sampled = sample_shard(local, math.ceil(per_class / count), SEED + index, enrich)
        for row in sampled:
            row["shard"] = shard
        rows.extend(sampled)
        print(shard, "selected:", len(sampled))
    return rows


def deduplicate(rows):
    by_hash, conflicting = {}, set()
    for row in rows:
        key = row["sha256"]
        if key in by_hash and by_hash[key]["label_id"] != row["label_id"]:
            conflicting.add(key)
        by_hash[key] = row
    return [r for key, r in by_hash.items() if key not in conflicting]


def choose_balanced(rows, per_class, seed, enrich=False):
    rng = random.Random(seed)
    result = []
    for label in (0, 1):
        group = [r for r in rows if r["label_id"] == label]
        rng.shuffle(group)
        result += select_coverage(group, [r for r in group if r["short"]],
                                  [r for r in group if r["is_login"] and label == 0], per_class,
                                  SHORT_FRACTION if enrich else 0,
                                  BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0)
    rng.shuffle(result)
    return result


def split_hosts(rows, val_per_class, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[host_group(row)].append(row)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    val, val_groups, counts = [], set(), Counter()
    for key in keys:
        group = groups[key]
        additions = Counter(r["label_id"] for r in group)
        if all(counts[label] + additions[label] <= val_per_class for label in (0, 1)):
            val.extend(group)
            counts.update(additions)
            val_groups.add(key)
    return [r for r in rows if host_group(r) not in val_groups], val


def assert_separate(splits):
    previous_hosts, previous_hashes = set(), set()
    for name, rows in splits.items():
        hosts, hashes = {host_group(r) for r in rows}, {r["sha256"] for r in rows}
        if hosts & previous_hosts or hashes & previous_hashes:
            raise ValueError(f"Host/content overlap in {name}")
        if len(hashes) != len(rows):
            raise ValueError(f"Duplicate content in {name}")
        if set(r["label_id"] for r in rows) != {0, 1}:
            raise ValueError(f"{name} needs both classes; increase shard/sample counts")
        previous_hosts.update(hosts)
        previous_hashes.update(hashes)


sample_config = {"version": 2, "dataset": DATASET_ID, "seed": SEED,
                 "train_shards": TRAIN_SHARDS, "test_shards": TEST_SHARDS,
                 "train_target_per_class": USE_TRAIN_PER_CLASS,
                 "val_target_per_class": USE_VAL_PER_CLASS,
                 "test_target_per_class": USE_TEST_PER_CLASS,
                 "train_pool_per_class": 2 * (USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS),
                 "test_pool_per_class": 3 * USE_TEST_PER_CLASS,
                 "min_chars": MIN_HTML_CHARS, "max_chars": MAX_HTML_CHARS,
                 "parse_budget": PARSE_CHAR_BUDGET, "short_chars": SHORT_HTML_CHARS,
                 "short_fraction": SHORT_FRACTION, "login_fraction": BENIGN_LOGIN_FRACTION}
cache_metadata = CACHE_DIR / "sampling.json"
cache_valid = (cache_metadata.exists()
               and json.loads(cache_metadata.read_text(encoding="utf-8")) == sample_config
               and all((CACHE_DIR / f"{name}.jsonl").exists() for name in ("pool", "test")))
if cache_valid:
    print("Reusing matching expanded sampling cache:", CACHE_DIR)
    def read_cache(name):
        with (CACHE_DIR / f"{name}.jsonl").open(encoding="utf-8") as handle:
            return [json.loads(line) for line in handle]
    pool, test_pool = read_cache("pool"), read_cache("test")
else:
    print("Building expanded sample; original small cache is preserved.")
    pool = sample_split("train", sample_config["train_pool_per_class"], TRAIN_SHARDS, True)
    test_pool = sample_split("test", sample_config["test_pool_per_class"], TEST_SHARDS, False)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in (("pool", pool), ("test", test_pool)):
        with (CACHE_DIR / f"{name}.jsonl").open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    cache_metadata.write_text(json.dumps(sample_config, indent=2), encoding="utf-8")

pool = deduplicate(pool)
train_pool, val_raw = split_hosts(pool, USE_VAL_PER_CLASS, SEED)
train_raw = choose_balanced(train_pool, USE_TRAIN_PER_CLASS, SEED, enrich=True)
known_hosts = {host_group(r) for r in train_raw + val_raw}
known_hashes = {r["sha256"] for r in pool}
test_pool = deduplicate(test_pool)
eligible_test = [r for r in test_pool if host_group(r) not in known_hosts and r["sha256"] not in known_hashes]
print("Excluded test pages with seen hosts/content:", len(test_pool) - len(eligible_test))
test_raw = choose_balanced(eligible_test, USE_TEST_PER_CLASS, SEED + 1)
assert_separate({"train": train_raw, "val": val_raw, "test": test_raw})
sampling_report = {}
targets = {"train": USE_TRAIN_PER_CLASS, "val": USE_VAL_PER_CLASS, "test": USE_TEST_PER_CLASS}
for name, rows in (("train", train_raw), ("val", val_raw), ("test", test_raw)):
    counts = Counter(r["label_id"] for r in rows)
    sampling_report[name] = {
        "target_per_class": targets[name], "actual_per_class": dict(counts),
        "actual_total": len(rows), "unique_hosts": len({r["host"] for r in rows}),
        "represented_shards": sorted({r["shard"] for r in rows}),
    }
    if any(counts[label] < targets[name] for label in (0, 1)):
        print(f"WARNING: {name} below target after deduplication/host isolation: {dict(counts)}. "
              "Increase shard coverage or candidate-pool multipliers before relying on target counts.")
    print(name, "labels:", dict(Counter(r["label_id"] for r in rows)),
          "short benign:", sum(r["short"] and r["label_id"] == 0 for r in rows),
          "benign login:", sum(r["is_login"] and r["label_id"] == 0 for r in rows))

## 3. Shared HTML preprocessing

Keep scripts, styles, noscript, templates and comments removed, with a 300,000-character parse budget.
Preserve cleaned markup, form destinations, input attributes and text as in the original baseline.
Inference must use these same settings and the saved question. A login form alone is not a phishing label.

In [ ]:
from bs4 import BeautifulSoup, Comment

STRIP_TAGS = ["script", "style", "noscript", "template"]


def clean_html(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(STRIP_TAGS):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    text = str(soup)
    if not text.strip():
        raise ValueError("No usable markup")
    return text


parse_failures = []


def build_pages(raw_examples, split):
    pages = []
    for item in raw_examples:
        try:
            pages.append({"text": clean_html(item["html"]), "label_id": item["label_id"],
                          "html_chars": len(item["html"]), "sha256": item["sha256"],
                          "host": item["host"], "shard": item["shard"], "short": item["short"], "is_login": item["is_login"]})
        except Exception as exc:
            parse_failures.append({"split": split, "sha256": item["sha256"], "reason": type(exc).__name__})
    return pages


train_pages = build_pages(train_raw, "train")
val_pages = build_pages(val_raw, "val")
test_pages = build_pages(test_raw, "test")
for name, pages in (("train", train_pages), ("val", val_pages), ("test", test_pages)):
    print(name, len(pages), dict(Counter(p["label_id"] for p in pages)))
print("parse failures:", dict(Counter(r["split"] for r in parse_failures)))

## 4. Encode and enforce the actual token budget

`build_sequence` may tokenize full HTML before slicing it. An intermediate tokenizer warning about
8,192 tokens is not evidence that those tokens reach the encoder: assertions below enforce the final
1,024-token sequence. The question and answer options share this budget with the HTML.
Each epoch gets a reproducible but different shuffle.

In [ ]:
from laya.common import build_sequence, collate_items, QTYPES

tokenizer_dir = snapshot_download(BASE_REPO, allow_patterns=["tokenizer/*"])
tok = AutoTokenizer.from_pretrained(os.path.join(tokenizer_dir, "tokenizer"))
PAD_ID = tok.pad_token_id
assert PAD_ID is not None
NOUL = QTYPES["noul"]

# Index 1 must be "true": laya/agent.py reports noul as p[1].
_probe_ids, _probe_markers = build_sequence(tok, "probe", QUESTION, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
assert len(_probe_markers) == 2, f"expected two options for a noul question, got {len(_probe_markers)}"


def to_item(page):
    ids, markers = build_sequence(tok, page["text"], QUESTION,
                                  max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
    assert 0 < len(ids) <= MAX_LEN
    assert len(markers) == 2 and all(0 <= m < len(ids) for m in markers)
    label = int(page["label_id"])          # 1 = phishing
    target = [0.0, 1.0] if label == 1 else [1.0, 0.0]
    return {"ids": ids, "markers": markers, "qtype": NOUL, "target": target, "label": label}


train_items = [to_item(p) for p in train_pages]
val_items = [to_item(p) for p in val_pages]
test_items = [to_item(p) for p in test_pages]
print(f"items: train {len(train_items)}  val {len(val_items)}  test {len(test_items)}")

_lengths = [len(it["ids"]) for it in train_items]
_capped = sum(l >= MAX_LEN for l in _lengths)
print(f"token length: median {sorted(_lengths)[len(_lengths)//2]}, "
      f"{_capped}/{len(_lengths)} ({_capped/len(_lengths):.0%}) hit the {MAX_LEN}-token cap")


def batches(items, size, shuffle=False, epoch=0):
    order = list(range(len(items)))
    if shuffle:
        random.Random(SEED + epoch).shuffle(order)
    for start in range(0, len(order), size):
        group = [[items[i]] for i in order[start:start + size]]
        yield collate_items(group, PAD_ID)

## 5. Load the checkpoint

`build_model` reconstructs the decision model from `rl_agent_config.json`, then the
published weights are loaded into it. Gradient checkpointing trades recompute for memory,
which is what lets a 421M-parameter encoder train on a 16 GB card.


In [ ]:
from laya.common import build_model
from safetensors.torch import load_file

model_dir = snapshot_download(BASE_REPO, allow_patterns=["model.safetensors", "encoder/*", "rl_agent_config.json"])
cfg = json.loads(Path(model_dir, "rl_agent_config.json").read_text())
print("encoder:", cfg["encoder"], "| head_layers:", cfg["head_layers"], "| shipped temps:", cfg.get("temperature"))

model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"), pretrained=False)
missing, unexpected = model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
if missing or unexpected:
    print("missing:", missing[:5], "| unexpected:", unexpected[:5])
assert MAX_LEN <= model.encoder.config.max_position_embeddings
model.encoder.config.reference_compile = False
model = model.to(device)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

total = sum(p.numel() for p in model.parameters())
print(f"parameters: {total/1e6:.0f}M  (MarkupLM-base is 136M)")

## 6. Train with warmup and correct accumulation

Encoder LR 1e-5; head LR 5e-5; effective batch 16. Warm up over 10% of planned optimizer updates,
then cosine decay. Losses are weighted by actual examples in each accumulation window, including
partial final windows. FP16 overflow skips do not advance the scheduler. Restore the checkpoint with
minimum validation cross-entropy; stop after two epochs without improvement.

Rerun the model-loading cell before starting a fresh training experiment.

### Recovering from interrupted training

Every improving **completed training + validation epoch** writes a complete Laya artifact under
`laya-phishing-v2-checkpoints/<run>/best-epoch-NNN/`. You can load it with Agent, but its metadata
marks it **uncalibrated**: temperature 1, default threshold 0.5, no test metrics. The final calibrated
artifact still goes to `laya-phishing-v2` after the later cells.

Every completed epoch also writes `last.pt` with current/best weights, optimizer, scheduler, scaler,
RNG states and progress. Set `RESUME_FROM` in configuration to its path, rerun setup/data/model cells,
and run training to resume at the next epoch. Identical hyperparameters, encoded data and package
versions are required. Work in an interrupted partial epoch is repeated; no mid-epoch resume is claimed.
An interrupted replacement preserves the previous complete `last.pt`.

If epoch 2 fails, epoch 1 is recoverable only after its save messages appeared. Before epoch 1's first
save, nothing is recoverable from disk. Updates to this notebook do not modify an already executing
Kaggle cell: apply them before starting a new run. Do not restart a live run expecting retroactive saves.

These are multi-GB files (model + best model + optimizer), and atomic replacements temporarily need
space for both checkpoint copies. Budget tens of GB including datasets and best-epoch folders. Completed
best artifacts are retained. Download/preserve the run folder through Kaggle outputs; session storage
alone is not an off-machine backup. No Hub upload is performed automatically.


In [ ]:
encoder_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    (encoder_params if name.startswith("encoder.") else head_params).append(param)
print(f"encoder tensors {len(encoder_params)} | head tensors {len(head_params)}")

optimizer = torch.optim.AdamW(
    [{"params": encoder_params, "lr": ENCODER_LR},
     {"params": head_params, "lr": HEAD_LR}], weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_items) / MICRO_BATCH / GRAD_ACCUM)
total_updates = steps_per_epoch * NUM_EPOCHS
warmup_updates = max(1, math.ceil(WARMUP_RATIO * total_updates))
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_updates, num_training_steps=total_updates)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
print(f"{steps_per_epoch} optimizer steps/epoch, effective batch {MICRO_BATCH * GRAD_ACCUM}")


def forward_loss(batch):
    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    assert batch["input_ids"].shape[-1] <= MAX_LEN
    logits, _ = model(batch["input_ids"], batch["attention_mask"],
                      batch["marker_pos"], batch["marker_mask"], batch["qtype"])
    logp = torch.log_softmax(logits.float().masked_fill(~batch["marker_mask"], -1e4), dim=-1)
    loss = -(batch["target"] * logp).sum(-1).mean()
    return loss, logits.detach(), batch["label"]


@torch.no_grad()
def evaluate(items):
    model.eval()
    total_loss, scores, labels = 0.0, [], []
    for batch in batches(items, MICRO_BATCH):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss, logits, label = forward_loss(batch)
        total_loss += loss.item() * len(label)
        scores.extend(torch.softmax(logits.float(), -1)[:, 1].cpu().tolist())
        labels.extend(label.cpu().tolist())
    predictions = [int(s >= 0.5) for s in scores]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1, zero_division=0)
    if not math.isfinite(total_loss) or not all(math.isfinite(s) for s in scores):
        raise RuntimeError("Non-finite validation results")
    return ({"loss": total_loss / len(labels), "accuracy": accuracy_score(labels, predictions),
             "precision_phishing": precision, "recall_phishing": recall, "f1_phishing": f1,
             "auc": roc_auc_score(labels, scores), "n": len(labels)}, scores, labels)


# Save at completed-epoch boundaries. The previous last.pt survives interrupted writes.
import uuid
from safetensors.torch import save_file


def atomic_torch_save(payload, destination):
    destination = Path(destination)
    temporary = destination.with_name(destination.name + ".tmp")
    with temporary.open("wb") as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)


def training_signature():
    # Include actual encoded inputs, not just counts: reject reordered/changed splits.
    digest = hashlib.sha256()
    for split, items in (("train", train_items), ("val", val_items), ("test", test_items)):
        digest.update(split.encode())
        for item in items:
            digest.update(json.dumps(item, sort_keys=True, separators=(",", ":")).encode())
    return {"data_sha256": digest.hexdigest(), "base": BASE_REPO,
            "max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN, "question": QUESTION,
            "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
            "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
            "epochs": NUM_EPOCHS, "warmup": WARMUP_RATIO, "seed": SEED,
            "amp_dtype": str(AMP_DTYPE), "patience": EARLY_STOPPING_PATIENCE,
            "clip": MAX_GRAD_NORM,
            "versions": {p: version(p) for p in ("torch", "transformers", "laya")}}


def save_best_artifact(epoch):
    destination = RUN_DIR / f"best-epoch-{epoch:03d}"
    temporary = RUN_DIR / f".best-epoch-{epoch:03d}-{uuid.uuid4().hex}.tmp"
    temporary.mkdir()
    save_file({k: v.float().contiguous() for k, v in best_state.items()},
              str(temporary / "model.safetensors"))
    model.encoder.config.save_pretrained(str(temporary / "encoder"))
    tok.save_pretrained(str(temporary / "tokenizer"))
    artifact_cfg = dict(cfg)
    # Base-model temperatures are not calibrated for the newly tuned weights.
    artifact_cfg.update(max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN,
                        model_name="laya-phishing-v2-recovery",
                        temperature=[1.0, 1.0, 1.0], temperature_by_options={})
    (temporary / "rl_agent_config.json").write_text(json.dumps(artifact_cfg, indent=2), encoding="utf-8")
    recovery_metadata = {
        "status": "uncalibrated_recovery", "best_epoch": epoch, "history": history,
        "base_model": BASE_REPO, "question": QUESTION,
        "id2label": {0: "benign", 1: "phishing"},
        "preprocessing": {"parse_char_budget": PARSE_CHAR_BUDGET, "strip_tags": STRIP_TAGS,
                          "strip_comments": True, "max_tokens": MAX_LEN,
                          "input_format": "cleaned HTML markup as text"},
        "inference": {"temperature": 1.0, "operating_threshold": 0.5,
                      "calibrated": False, "threshold_selected": False},
        "test_metrics": {}, "training_signature": run_signature,
        "note": "Emergency inference artifact only. Calibration, test and reload parity are pending."}
    (temporary / "metadata.json").write_text(json.dumps(recovery_metadata, indent=2), encoding="utf-8")
    os.replace(temporary, destination)
    print("Saved usable but UNCALIBRATED best artifact:", destination.resolve(), flush=True)


def save_resume_checkpoint(epoch):
    numpy_state = np.random.get_state()
    payload = {
        "format_version": 1, "completed_epoch": epoch, "signature": run_signature,
        "model": model.state_dict(), "best_model": best_state,
        "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(), "history": history,
        "best_loss": best_loss, "best_epoch": best_epoch, "stale": stale,
        "optimizer_updates": optimizer_updates, "python_rng": random.getstate(),
        "numpy_rng": (numpy_state[0], numpy_state[1].tolist(), *numpy_state[2:]),
        "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
    }
    atomic_torch_save(payload, RUN_DIR / "last.pt")
    print("Saved resumable checkpoint:", (RUN_DIR / "last.pt").resolve(), flush=True)


run_signature = training_signature()
# New directory also when resuming: never overwrite a mounted/read-only checkpoint.
RUN_DIR = CHECKPOINT_ROOT / (time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Checkpoint directory:", RUN_DIR.resolve(), flush=True)
history, best_state, best_loss, best_epoch, stale = [], None, float("inf"), None, 0
def accumulation_examples(step_index, n_items, micro_batch, grad_accum):
    """Number of examples in the zero-based step's accumulation window."""
    start = (step_index // grad_accum) * grad_accum * micro_batch
    return min(grad_accum * micro_batch, n_items - start)

optimizer_updates = 0
start_epoch = 1
if RESUME_FROM is not None:
    checkpoint = torch.load(Path(RESUME_FROM), map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != 1 or checkpoint["signature"] != run_signature:
        raise ValueError("Checkpoint config/data/version mismatch. Restore the original settings and splits.")
    model.load_state_dict(checkpoint["model"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    history, best_state = checkpoint["history"], checkpoint["best_model"]
    best_loss, best_epoch, stale = checkpoint["best_loss"], checkpoint["best_epoch"], checkpoint["stale"]
    optimizer_updates = checkpoint["optimizer_updates"]
    start_epoch = checkpoint["completed_epoch"] + 1
    random.setstate(checkpoint["python_rng"])
    rng = checkpoint["numpy_rng"]
    np.random.set_state((rng[0], np.array(rng[1], dtype=np.uint32), *rng[2:]))
    torch.set_rng_state(checkpoint["torch_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    del checkpoint
    print("Resuming at epoch", start_epoch, "with best epoch", best_epoch, flush=True)
    if stale >= EARLY_STOPPING_PATIENCE:
        start_epoch = NUM_EPOCHS + 1  # completed early stopping; continue to calibration
for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print("Running Epoch", epoch)
    model.train()
    running, seen, step = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    started = time.time()
    for batch in batches(train_items, MICRO_BATCH, shuffle=True, epoch=epoch):
        step += 1
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss, _, label = forward_loss(batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at epoch {epoch} step {step}")
        window_examples = accumulation_examples(step - 1, len(train_items), MICRO_BATCH, GRAD_ACCUM)
        scaler.scale(loss * len(label) / window_examples).backward()
        if step % GRAD_ACCUM == 0 or step * MICRO_BATCH >= len(train_items):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
                optimizer_updates += 1
            optimizer.zero_grad(set_to_none=True)
        running += loss.item() * len(label); seen += len(label)
        if step % 100 == 0:
            elapsed = time.time() - started
            remaining = math.ceil(len(train_items) / MICRO_BATCH) - step
            print(f"Epoch {epoch}: batch {step}/{math.ceil(len(train_items) / MICRO_BATCH)}; "
                  f"training ETA {remaining * elapsed / step / 60:.1f} min (validation extra)", flush=True)
    print("Training passes complete; running validation", flush=True)
    val_metrics, _, _ = evaluate(val_items)
    history.append({"epoch": epoch, "train_loss": running / seen, "val": val_metrics,
                    "optimizer_updates": optimizer_updates, "seconds": round(time.time() - started)})
    print(json.dumps(history[-1]))
    if val_metrics["loss"] < best_loss:
        best_loss, best_epoch, stale = val_metrics["loss"], epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  best so far (epoch {epoch}, val_loss {best_loss:.4f})")
        save_best_artifact(epoch)
    else:
        stale += 1
    save_resume_checkpoint(epoch)
    if stale >= EARLY_STOPPING_PATIENCE:
        print("early stopping: validation loss stopped improving")
        break

if best_state is None:
    raise RuntimeError("no checkpoint was produced")
model.load_state_dict(best_state)
model.eval()
print("restored epoch", best_epoch)

## 7. Validation-only calibration and operating threshold

Fit a bounded temperature in [0.5, 5.0], matching Laya Agent. Keep temperature 1 if calibration
would worsen validation NLL. Then maximize validation recall subject to the configured empirical FPR
limit, handling tied scores using the same `>=` decision rule as inference. Neither step reads test.
Temperature changes probabilities but, by itself, does not change binary decisions at threshold 0.5.

In [ ]:
_, val_scores, val_labels = evaluate(val_items)


@torch.no_grad()
def collect_logits(items):
    model.eval()
    out = []
    for batch in batches(items, MICRO_BATCH):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits, _ = model(batch["input_ids"], batch["attention_mask"],
                              batch["marker_pos"], batch["marker_mask"], batch["qtype"])
        out.append(logits.float().cpu())
    return torch.cat(out)


val_logits = collect_logits(val_items)
val_target = torch.tensor([[0.0, 1.0] if it["label"] == 1 else [1.0, 0.0] for it in val_items])

# Parameterization keeps optimization inside the exact runtime-supported range.
initial_fraction = (1.0 - TEMPERATURE_MIN) / (TEMPERATURE_MAX - TEMPERATURE_MIN)
raw_t = torch.tensor([math.log(initial_fraction / (1 - initial_fraction))], requires_grad=True)
def temperature_value():
    return TEMPERATURE_MIN + (TEMPERATURE_MAX - TEMPERATURE_MIN) * raw_t.sigmoid()
lbfgs = torch.optim.LBFGS([raw_t], lr=0.1, max_iter=100)


def closure():
    lbfgs.zero_grad()
    loss = -(val_target * torch.log_softmax(val_logits / temperature_value(), -1)).sum(-1).mean()
    loss.backward()
    return loss


lbfgs.step(closure)
TEMPERATURE = float(temperature_value().item())


def nll(logits, target, temperature):
    return float(-(target * torch.log_softmax(logits / temperature, -1)).sum(-1).mean())


if not math.isfinite(TEMPERATURE) or nll(val_logits, val_target, TEMPERATURE) > nll(val_logits, val_target, 1.0):
    TEMPERATURE = 1.0

print(f"fitted temperature {TEMPERATURE:.4f}")
print(f"  validation NLL  {nll(val_logits, val_target, 1.0):.4f} -> {nll(val_logits, val_target, TEMPERATURE):.4f}")


def select_threshold(scores, labels, target_fpr):
    if len(scores) != len(labels) or set(labels) != {0, 1}:
        raise ValueError("Threshold selection requires aligned data with both classes")
    if not 0 <= target_fpr <= 1 or not all(math.isfinite(s) and 0 <= s <= 1 for s in scores):
        raise ValueError("Invalid FPR or probability")
    negatives, positives = labels.count(0), labels.count(1)
    candidates = sorted({0.0, 0.5, 1.0, *scores,
                         *(math.nextafter(s, math.inf) for s in scores if s < 1.0)})
    best = None
    for threshold in candidates:
        fp = sum(y == 0 and s >= threshold for s, y in zip(scores, labels))
        tp = sum(y == 1 and s >= threshold for s, y in zip(scores, labels))
        if fp / negatives > target_fpr:
            continue
        key = (tp / positives, -fp / negatives, threshold)
        if best is None or key > best[0]:
            best = (key, threshold, fp, tp)
    if best is None:
        raise RuntimeError("No threshold in [0, 1] satisfies the FPR target; inspect saturated scores")
    return best[1], {"target_fpr": target_fpr, "fpr": best[2] / negatives,
                     "recall": best[3] / positives, "benign_n": negatives,
                     "phishing_n": positives, "fp": best[2], "tp": best[3]}

calibrated_val_scores = torch.softmax(val_logits.double() / TEMPERATURE, -1)[:, 1].tolist()
OPERATING_THRESHOLD, threshold_selection = select_threshold(
    calibrated_val_scores, val_labels, TARGET_FPR)
print("Validation threshold:", OPERATING_THRESHOLD, threshold_selection)
print("Empirical FPR resolution:", 1 / threshold_selection["benign_n"])

## 8. Held-out test

Apply the already selected temperature and threshold. Report both the operating point and 0.5,
plus AUC, FPR and page-length baseline. Do not tune on these results. Historical MarkupLM/Mendeley
numbers are omitted because identical example membership has not been established.

In [ ]:
test_logits = collect_logits(test_items)
test_scores = torch.softmax(test_logits.double() / TEMPERATURE, -1)[:, 1].tolist()
test_labels = [it["label"] for it in test_items]

def summarize_scores(labels, scores, threshold):
    predictions = [int(s >= threshold) for s in scores]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(labels, predictions),
            "precision_phishing": precision, "recall_phishing": recall, "f1_phishing": f1,
            "auc": roc_auc_score(labels, scores) if len(set(labels)) == 2 else None,
            "fpr": float(fp / (tn + fp)) if tn + fp else None,
            "confusion": [[int(tn), int(fp)], [int(fn), int(tp)]],
            "threshold": threshold, "n": len(labels)}

test_metrics = summarize_scores(test_labels, test_scores, OPERATING_THRESHOLD)
test_metrics["temperature"] = TEMPERATURE
test_metrics_at_half = summarize_scores(test_labels, test_scores, 0.5)
length_baseline = roc_auc_score(test_labels, [-p["html_chars"] for p in test_pages])
print("Operating threshold:", json.dumps(test_metrics, indent=2))
print("At 0.5:", json.dumps(test_metrics_at_half, indent=2))
print("Page-length-only AUC:", length_baseline)
print("Model minus length AUC:", test_metrics["auc"] - length_baseline)

slice_metrics = {}
for name, indices in {
    "short_benign": [i for i,p in enumerate(test_pages) if p["short"] and p["label_id"] == 0],
    "benign_login": [i for i,p in enumerate(test_pages) if p["is_login"] and p["label_id"] == 0],
    "long_pages": [i for i,p in enumerate(test_pages) if not p["short"]],
}.items():
    slice_metrics[name] = (summarize_scores([test_labels[i] for i in indices],
        [test_scores[i] for i in indices], OPERATING_THRESHOLD) if indices else {"n": 0})
print("Diagnostic slices:", json.dumps(slice_metrics, indent=2))

## 9. Save the complete inference contract

Save float32 weights to avoid adding export quantization drift, both sequence budgets, temperature,
threshold, preprocessing, question, package versions and a split manifest. Float32 increases artifact size.
The Agent returns probabilities; the caller must load `metadata.json` and apply the saved operating
threshold instead of hard-coding 0.5. Use the same cleaning function and question at inference.

In [ ]:
from safetensors.torch import save_file

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
state = {k: v.detach().float().contiguous().cpu() for k, v in model.state_dict().items()}
save_file(state, str(ARTIFACT_DIR / "model.safetensors"))
model.encoder.config.save_pretrained(str(ARTIFACT_DIR / "encoder"))
tok.save_pretrained(str(ARTIFACT_DIR / "tokenizer"))

out_cfg = dict(cfg)
out_cfg["max_len"] = MAX_LEN
out_cfg["head_max_len"] = HEAD_MAX_LEN
# temperature is indexed by question type: [choice, score, noul]
temps = list(out_cfg.get("temperature", [1.0, 1.0, 1.0]))
temps[QTYPES["noul"]] = TEMPERATURE
out_cfg["temperature"] = temps
out_cfg["temperature_by_options"] = {**out_cfg.get("temperature_by_options", {}), "noul:2": TEMPERATURE}
out_cfg["model_name"] = "laya-phishing-v2"
(ARTIFACT_DIR / "rl_agent_config.json").write_text(json.dumps(out_cfg, indent=2))

metadata = {
    "base_model": BASE_REPO,
    "task": "sequence-classification",
    "id2label": {0: "benign", 1: "phishing"},
    "train_dataset": "phreshphish/phreshphish",
    "seed": SEED,
    "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
    "effective_batch_size": MICRO_BATCH * GRAD_ACCUM,
    "max_epochs": NUM_EPOCHS, "best_epoch": best_epoch,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "selection_metric": "lowest validation cross-entropy",
    "history": history,
    "train_n": len(train_items), "val_n": len(val_items), "test_n": len(test_items),
    "test_metrics": test_metrics,
    "preprocessing": {"parse_char_budget": PARSE_CHAR_BUDGET,
                      "strip_tags": STRIP_TAGS, "strip_comments": True,
                      "max_tokens": MAX_LEN, "input_format": "cleaned HTML markup as text"},
    "inference": {"operating_threshold": OPERATING_THRESHOLD, "temperature": TEMPERATURE},
    "question": QUESTION,
    "baselines": {"length_only_auc_test": length_baseline},
    "objective": "cross-entropy, matching train_html_phishing_classifier.ipynb; "
                 "not Laya's RLCD reinforcement objective",
    "note": "Expanded shard coverage with exact-host isolation; compare using saved split manifests.",
    "sampling_config": sample_config, "sampling_report": sampling_report,
    "head_max_len": HEAD_MAX_LEN, "warmup_ratio": WARMUP_RATIO,
    "optimizer_updates": optimizer_updates, "target_fpr": TARGET_FPR,
    "threshold_selection": threshold_selection, "test_metrics_at_half": test_metrics_at_half,
    "slice_metrics": slice_metrics,
    "versions": {p: version(p) for p in ("torch", "transformers", "laya", "safetensors")},
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
print("saved to", ARTIFACT_DIR.resolve())
for item in sorted(ARTIFACT_DIR.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(ARTIFACT_DIR)}  {item.stat().st_size/1e6:.1f} MB")
with (ARTIFACT_DIR / "split_manifest.jsonl").open("w", encoding="utf-8") as handle:
    for split, pages in (("train", train_pages), ("val", val_pages), ("test", test_pages)):
        for p in pages:
            handle.write(json.dumps({k: p[k] for k in ("sha256", "host", "shard", "label_id", "short", "is_login")}
                                   | {"split": split}) + "\n")
(ARTIFACT_DIR / "parse_failures.json").write_text(json.dumps(parse_failures, indent=2), encoding="utf-8")

## 10. Reload through the inference Agent

Check a short page and a long page using the exact exported weights and question. Compare against
float32 CPU inference before export reload, avoiding GPU autocast differences. This is a parity check,
not a new evaluation or threshold-tuning step. It may take time on CPU. The artifact is only verified
when this cell prints PASS. Run inference notebooks with this artifact and its saved metadata.

In [ ]:
import gc
from laya import Agent

# Release optimizer GPU state before loading the exported Agent.
del optimizer, scheduler, scaler, best_state, state
model = model.float().cpu().eval()
model.encoder.gradient_checkpointing_disable()
if hasattr(model.encoder.config, "reference_compile"):
    model.encoder.config.reference_compile = False
torch.cuda.empty_cache()
gc.collect()
probe_indices = sorted({min(range(len(val_items)), key=lambda i: len(val_items[i]["ids"])),
                        max(range(len(val_items)), key=lambda i: len(val_items[i]["ids"]))})
reference_scores = []
with torch.inference_mode():
    for i in probe_indices:
        b = collate_items([[val_items[i]]], PAD_ID)
        logits, _ = model(b["input_ids"], b["attention_mask"], b["marker_pos"], b["marker_mask"], b["qtype"])
        reference_scores.append(torch.softmax(logits.double() / TEMPERATURE, -1)[0, 1].item())
del model
gc.collect()
agent = Agent(str(ARTIFACT_DIR.resolve()), device="cpu")
assert agent.cfg["max_len"] == MAX_LEN
assert agent.cfg["head_max_len"] == HEAD_MAX_LEN
sdk_question = {"phishing": {"type": QUESTION["t"], "instructions": QUESTION["ins"]}}
parity = []
for i, expected in zip(probe_indices, reference_scores):
    result = agent.predict(val_pages[i]["text"], sdk_question)
    actual = float(result["answers"]["phishing"]["noul"])
    tokens = int(result["usage"]["input_tokens"])
    assert tokens == len(val_items[i]["ids"]) and tokens <= MAX_LEN, (tokens, len(val_items[i]["ids"]))
    assert math.isclose(actual, expected, rel_tol=1e-4, abs_tol=1e-4), (actual, expected)
    parity.append({"input_tokens": tokens, "reference": expected, "reloaded": actual})
(ARTIFACT_DIR / "reload_parity.json").write_text(json.dumps(parity, indent=2), encoding="utf-8")
print("PASS: exported Agent matches training inputs and calibrated probabilities", parity)